### 1. Conexion

In [31]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
assert DATABASE_URL is not None, "No se encontró DATABASE_URL"

engine = create_engine(DATABASE_URL)

SCHEMA = "dm_enaho2025"

### 2. Consultar las columnas, llaves y sus tipos

In [32]:
query_columns = """
SELECT
    c.table_name,
    c.column_name,
    c.ordinal_position,
    c.data_type,
    c.is_nullable
FROM information_schema.columns c
WHERE c.table_schema = :schema
ORDER BY c.table_name, c.ordinal_position;
"""

query_pk = """
SELECT
    tc.table_name,
    kcu.column_name
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
   AND tc.table_schema = kcu.table_schema
WHERE tc.table_schema = :schema
  AND tc.constraint_type = 'PRIMARY KEY'
ORDER BY tc.table_name, kcu.ordinal_position;
"""

query_fk = """
SELECT
    tc.table_name AS child_table,
    kcu.column_name AS child_column,
    ccu.table_name AS parent_table,
    ccu.column_name AS parent_column,
    tc.constraint_name
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name
   AND tc.table_schema = kcu.table_schema
JOIN information_schema.constraint_column_usage ccu
    ON ccu.constraint_name = tc.constraint_name
   AND ccu.table_schema = tc.table_schema
WHERE tc.table_schema = :schema
  AND tc.constraint_type = 'FOREIGN KEY'
ORDER BY child_table, constraint_name;
"""

columns_df = pd.read_sql(text(query_columns), engine, params={"schema": SCHEMA})
pk_df = pd.read_sql(text(query_pk), engine, params={"schema": SCHEMA})
fk_df = pd.read_sql(text(query_fk), engine, params={"schema": SCHEMA})

print("Columnas:", columns_df.shape)
print("PK:", pk_df.shape)
print("FK:", fk_df.shape)

columns_df.head()

Columnas: (129, 5)
PK: (11, 2)
FK: (10, 5)


,table_name,column_name,ordinal_position,data_type,is_nullable
0,dim_agua,agua_key,1,bigint,NO
1,dim_agua,fuente_agua_cod,2,bigint,YES
2,dim_agua,fuente_agua,3,text,YES
3,dim_agua,nivel_cloro_cod,4,bigint,YES
4,dim_agua,nivel_cloro,5,text,YES


### 3. Estructuras auxiliares

In [33]:
pk_map = (
    pk_df.groupby("table_name")["column_name"]
    .apply(set)
    .to_dict()
)

fk_map = (
    fk_df.groupby("child_table")["column_name"] if "column_name" in fk_df.columns else
    fk_df.groupby("child_table")["child_column"]
    .apply(set)
    .to_dict()
)

# Corrección robusta por si fk_map no se construyó arriba
fk_map = (
    fk_df.groupby("child_table")["child_column"]
    .apply(set)
    .to_dict()
)

columns_by_table = (
    columns_df.groupby("table_name")
    .apply(lambda x: x[["column_name", "data_type", "is_nullable"]].to_dict("records"))
    .to_dict()
)

tables = sorted(columns_df["table_name"].unique())

tables

C:\Users\guill\AppData\Local\Temp\ipykernel_5812\2010631319.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[["column_name", "data_type", "is_nullable"]].to_dict("records"))


['dim_agua',
 'dim_conectividad',
 'dim_energia',
 'dim_geografia',
 'dim_habitabilidad',
 'dim_materiales_vivienda',
 'dim_saneamiento',
 'dim_tenencia_propiedad',
 'dim_tiempo',
 'dim_vivienda',
 'fact_hogar_bienestar']

### 4. Visualización

In [42]:
MODO = "completo"   # opciones: "resumen", "completo"

### 5. Columnas por tabla

In [43]:
priority_columns = {
    "fact_hogar_bienestar": [
        "hogar_id",
        "tiempo_key",
        "geografia_key",
        "vivienda_key",
        "materiales_vivienda_key",
        "habitabilidad_key",
        "tenencia_propiedad_key",
        "agua_key",
        "saneamiento_key",
        "energia_key",
        "conectividad_key",
        "factor_expansion",
        "total_carencias_bienestar_habitacional",
        "brecha_multidimensional_ind"
    ],
    "dim_tiempo": [
        "tiempo_key",
        "anio",
        "mes",
        "periodo"
    ],
    "dim_geografia": [
        "geografia_key",
        "ubigeo",
        "dominio_cod",
        "dominio",
        "estrato_cod",
        "estrato",
        "latitud",
        "longitud",
        "altitud"
    ],
    "dim_vivienda": [
        "vivienda_key",
        "tipo_vivienda_cod",
        "tipo_vivienda"
    ],
    "dim_materiales_vivienda": [
        "materiales_vivienda_key",
        "material_pared_cod",
        "material_pared",
        "material_piso_cod",
        "material_piso",
        "material_techo_cod",
        "material_techo",
        "vivienda_material_precario_ind"
    ],
    "dim_habitabilidad": [
        "habitabilidad_key",
        "rango_habitaciones_total",
        "rango_habitaciones_dormir",
        "nbi_vivienda_inadecuada_ind",
        "nbi_hacinamiento_ind",
        "categoria_carencias",
        "brecha_multidimensional_ind"
    ],
    "dim_tenencia_propiedad": [
        "tenencia_propiedad_key",
        "tenencia_vivienda_cod",
        "tenencia_vivienda"
    ],
    "dim_agua": [
        "agua_key",
        "fuente_agua_cod",
        "fuente_agua",
        "nivel_cloro_cod",
        "nivel_cloro",
        "agua_red_publica_ind",
        "agua_potable_ind",
        "agua_todos_dias_ind",
        "agua_segura_ind"
    ],
    "dim_saneamiento": [
        "saneamiento_key",
        "servicio_higienico_cod",
        "servicio_higienico",
        "saneamiento_adecuado_ind"
    ],
    "dim_energia": [
        "energia_key",
        "combustible_cocina_cod",
        "combustible_cocina",
        "tiene_electricidad_ind",
        "combustible_limpio_ind",
        "combustible_solido_ind"
    ],
    "dim_conectividad": [
        "conectividad_key",
        "tiene_telefono_fijo_ind",
        "tiene_celular_ind",
        "tiene_tv_cable_ind",
        "tiene_internet_ind",
        "sin_tic_ind",
        "tiene_tdt_ind"
    ]
}

### 5. Funciones para crear el diagrama

In [44]:
def pretty_type(dtype: str) -> str:
    dtype = str(dtype).lower()

    mapping = {
        "text": "TEXT",
        "character varying": "VARCHAR",
        "integer": "INT",
        "bigint": "BIGINT",
        "double precision": "DOUBLE",
        "numeric": "NUMERIC",
        "boolean": "BOOL"
    }
    return mapping.get(dtype, dtype.upper())


def get_columns_for_display(table_name):
    cols = columns_by_table[table_name]

    if MODO == "completo":
        return cols

    priority = priority_columns.get(table_name, [])
    priority_set = set(priority)

    selected = [c for c in cols if c["column_name"] in priority_set]

    selected_sorted = []
    for p in priority:
        for c in selected:
            if c["column_name"] == p:
                selected_sorted.append(c)

    return selected_sorted


def build_html_label(table_name):
    pk_cols = pk_map.get(table_name, set())
    fk_cols = fk_map.get(table_name, set())
    cols = get_columns_for_display(table_name)

    header_bg = "#0B2E63"
    header_font = "#FFFFFF"
    row_bg = "#F8FBFF"
    border_color = "#1F3A5F"

    html = []
    html.append(f'''<
    <TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="6" COLOR="{border_color}">
        <TR>
            <TD BGCOLOR="{header_bg}" COLSPAN="3">
                <FONT COLOR="{header_font}"><B>{table_name}</B></FONT>
            </TD>
        </TR>
        <TR>
            <TD BGCOLOR="#DCE9F9"><B>Clave</B></TD>
            <TD BGCOLOR="#DCE9F9"><B>Columna</B></TD>
            <TD BGCOLOR="#DCE9F9"><B>Tipo</B></TD>
        </TR>
    ''')

    for col in cols:
        col_name = col["column_name"]
        dtype = pretty_type(col["data_type"])

        key_label = ""
        if col_name in pk_cols and col_name in fk_cols:
            key_label = "PK/FK"
        elif col_name in pk_cols:
            key_label = "PK"
        elif col_name in fk_cols:
            key_label = "FK"

        font_open = '<FONT FACE="Helvetica">'
        font_close = '</FONT>'

        if col_name in pk_cols:
            font_open = '<FONT FACE="Helvetica"><B>'
            font_close = '</B></FONT>'

        html.append(f'''
        <TR>
            <TD BGCOLOR="{row_bg}" ALIGN="CENTER">{key_label}</TD>
            <TD BGCOLOR="{row_bg}" ALIGN="LEFT">{font_open}{col_name}{font_close}</TD>
            <TD BGCOLOR="{row_bg}" ALIGN="LEFT"><FONT FACE="Helvetica" POINT-SIZE="10">{dtype}</FONT></TD>
        </TR>
        ''')

    if MODO == "resumen":
        total_real = len(columns_by_table[table_name])
        total_mostradas = len(cols)
        if total_real > total_mostradas:
            html.append(f'''
            <TR>
                <TD BGCOLOR="#F1F1F1" COLSPAN="3" ALIGN="CENTER">
                    <FONT POINT-SIZE="10"><I>+ {total_real - total_mostradas} columnas adicionales</I></FONT>
                </TD>
            </TR>
            ''')

    html.append("</TABLE>>")
    return "".join(html)

In [45]:
dot = Digraph("ERD_ENAHO_2025", format="png")

# Layout general corregido
dot.attr(
    rankdir="TB",       # de arriba hacia abajo
    splines="ortho",
    nodesep="0.45",
    ranksep="0.75",
    pad="0.3",
    bgcolor="white"
)

dot.attr("node", shape="plaintext", fontname="Helvetica")
dot.attr("edge", color="#6B7280", penwidth="1.2", arrowsize="0.7")

fact_table = "fact_hogar_bienestar"

fila_superior = [
    "dim_tiempo",
    "dim_geografia",
    "dim_vivienda",
    "dim_materiales_vivienda",
    "dim_habitabilidad"
]

fila_inferior = [
    "dim_tenencia_propiedad",
    "dim_agua",
    "dim_saneamiento",
    "dim_energia",
    "dim_conectividad"
]

fila_superior = [t for t in fila_superior if t in tables]
fila_inferior = [t for t in fila_inferior if t in tables]

# Fila superior
with dot.subgraph(name="cluster_top") as top:
    top.attr(rank="same")
    top.attr(color="white")
    for table in fila_superior:
        top.node(table, label=build_html_label(table))

# Fila central
with dot.subgraph(name="cluster_middle") as middle:
    middle.attr(rank="same")
    middle.attr(color="white")
    middle.node(fact_table, label=build_html_label(fact_table))

# Fila inferior
with dot.subgraph(name="cluster_bottom") as bottom:
    bottom.attr(rank="same")
    bottom.attr(color="white")
    for table in fila_inferior:
        bottom.node(table, label=build_html_label(table))

# Relaciones FK
for _, row in fk_df.iterrows():
    child = row["child_table"]
    parent = row["parent_table"]
    child_col = row["child_column"]

    # relación simple, más limpia
    dot.edge(parent, child)

    # si quieres mostrar el nombre de la FK, reemplaza la línea de arriba por esta:
    # dot.edge(parent, child, xlabel=child_col, fontsize="9", fontname="Helvetica")

### 6. Renderizar y guardar el diagrama

In [46]:
OUTPUT_DIR = Path.cwd().parent / "data" / "outputs" / "diagrams"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

nombre_base = f"erd_enaho2025_{MODO}_compacto"

path_render = dot.render(
    filename=nombre_base,
    directory=str(OUTPUT_DIR),
    cleanup=True
)

print("Diagrama generado en:")
print(path_render)

Diagrama generado en:
g:\Mi unidad\UP - Ingeniería de la información\Semestre IX\Business Intelligence\TrabajoFinal\repo\data\outputs\diagrams\erd_enaho2025_completo_compacto.png
